# Genova Quickstart Guide

This notebook demonstrates the core capabilities of the Genova genomics foundation model:

1. Loading the model
2. Tokenizing DNA sequences
3. Extracting embeddings
4. Predicting variant effects

**No real genome files are needed** -- we use synthetic sequences throughout.

## 1. Setup and Imports

In [ ]:
import numpy as np
import torch

from genova.utils.config import GenovaConfig, ModelConfig
from genova.data.tokenizer import GenomicTokenizer
from genova.models.model_factory import create_model, count_parameters

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Create a Tokenizer

Genova uses k-mer tokenization to convert DNA sequences into integer tokens.
The default k-mer size is 6, which gives us 4^6 = 4096 possible tokens.

In [ ]:
# Create a k-mer tokenizer with k=6
tokenizer = GenomicTokenizer(mode="kmer", k=6, stride=1)
tokenizer.build_vocab()

print(f"Vocabulary size: {tokenizer.vocab_size}")

# Tokenize a synthetic DNA sequence
sequence = "ACGTACGTACGTACGTNNNNACGTACGT"
tokens = tokenizer.encode(sequence)
print(f"Input sequence: {sequence}")
print(f"Token IDs: {tokens[:10]}... (total: {len(tokens)})")

# Decode back
decoded = tokenizer.decode(tokens)
print(f"Decoded: {decoded}")

**Expected output:**
```
Vocabulary size: 4102  (4096 k-mers + special tokens)
Input sequence: ACGTACGTACGTACGTNNNNACGTACGT
Token IDs: [123, 456, ...] (total: 23)
```

## 3. Load or Create a Model

We create a small model for demonstration. In production you would load
from a pre-trained checkpoint.

In [ ]:
# Configure a small model for demo
config = GenovaConfig()
config.model.d_model = 128
config.model.n_heads = 4
config.model.n_layers = 2
config.model.d_ff = 512
config.model.vocab_size = tokenizer.vocab_size

# Create model
model = create_model(config.model, task="backbone")
model.eval()

n_params = count_parameters(model)
print(f"Model parameters: {n_params:,}")
print(f"Architecture: {config.model.arch}")
print(f"d_model={config.model.d_model}, n_layers={config.model.n_layers}, n_heads={config.model.n_heads}")

**Expected output:**
```
Model parameters: ~500,000
Architecture: transformer
d_model=128, n_layers=2, n_heads=4
```

## 4. Extract Embeddings

Embeddings capture the learned representation of a DNA sequence.

In [ ]:
# Prepare input
sequences = [
    "ACGTACGTACGTACGTACGTACGTACGT",
    "TTTTAAAACCCCGGGGTTTTAAAACCCC",
    "GCGCGCGCATATATATNNNNGCGCGCGC",
]

encoded = tokenizer.batch_encode(sequences, max_length=128, padding=True)
input_ids = torch.tensor(encoded["input_ids"], dtype=torch.long)
attention_mask = torch.tensor(encoded["attention_mask"], dtype=torch.long)

print(f"Input shape: {input_ids.shape}")

# Forward pass
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

if isinstance(outputs, dict):
    hidden = outputs["last_hidden_state"]
else:
    hidden = outputs

print(f"Hidden states shape: {hidden.shape}")

# Mean pooling
mask = attention_mask.unsqueeze(-1).float()
embeddings = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
print(f"Embedding shape: {embeddings.shape}")
print(f"Embedding norm: {torch.norm(embeddings, dim=-1).tolist()}")

**Expected output:**
```
Input shape: torch.Size([3, 128])
Hidden states shape: torch.Size([3, 128, 128])
Embedding shape: torch.Size([3, 128])
Embedding norm: [~5.0, ~5.0, ~5.0]
```

## 5. Predict Variant Effects

Compare reference and alternate allele embeddings to score variant pathogenicity.

In [ ]:
# Simulate a variant: reference vs alternate
ref_seq = "ACGTACGTACGTACGTACGTACGTACGT"
alt_seq = "ACGTACGTACGTTCGTACGTACGTACGT"  # T>T (SNV at position 13)

# Encode both
ref_encoded = tokenizer.batch_encode([ref_seq], max_length=128, padding=True)
alt_encoded = tokenizer.batch_encode([alt_seq], max_length=128, padding=True)

ref_ids = torch.tensor(ref_encoded["input_ids"], dtype=torch.long)
alt_ids = torch.tensor(alt_encoded["input_ids"], dtype=torch.long)
ref_mask = torch.tensor(ref_encoded["attention_mask"], dtype=torch.long)
alt_mask = torch.tensor(alt_encoded["attention_mask"], dtype=torch.long)

with torch.no_grad():
    ref_out = model(input_ids=ref_ids, attention_mask=ref_mask)
    alt_out = model(input_ids=alt_ids, attention_mask=alt_mask)

ref_hidden = ref_out["last_hidden_state"] if isinstance(ref_out, dict) else ref_out
alt_hidden = alt_out["last_hidden_state"] if isinstance(alt_out, dict) else alt_out

# Pool and compare
ref_emb = ref_hidden.mean(dim=1)
alt_emb = alt_hidden.mean(dim=1)

diff = (alt_emb - ref_emb).squeeze()
score = float(torch.norm(diff).item())
pathogenicity = 1.0 / (1.0 + np.exp(-score + 2.0))  # sigmoid

print(f"Variant effect score (L2 norm): {score:.4f}")
print(f"Pathogenicity probability: {pathogenicity:.4f}")
print(f"Prediction: {'pathogenic' if pathogenicity >= 0.5 else 'benign'}")

**Expected output:**
```
Variant effect score (L2 norm): ~1.5
Pathogenicity probability: ~0.38
Prediction: benign
```

(Actual values depend on random model initialisation.)

## 6. Using the Inference Engine

For production use, the `InferenceEngine` wraps model loading, tokenization, and batched inference.

In [ ]:
from genova.api.inference import InferenceEngine

# In production you'd pass model_path to a checkpoint directory.
# Here we create an engine with default config for demonstration.
engine = InferenceEngine(device="cpu", max_batch_size=8)
engine.load()

# Embed sequences
embeddings = engine.embed(["ACGTACGTACGT", "TTTTAAAACCCC"])
print(f"Embedding 1 shape: {embeddings[0].shape}")
print(f"Embedding 2 shape: {embeddings[1].shape}")

# Variant prediction
results = engine.predict_variant(
    ref_sequences=["ACGTACGTACGTACGTACGT"],
    alt_sequences=["ACGTACGTACTTACGTACGT"],
)
print(f"Variant prediction: {results[0]}")

**Expected output:**
```
Embedding 1 shape: (768,)
Embedding 2 shape: (768,)
Variant prediction: {'score': 0.73, 'label': 'pathogenic', 'confidence': 0.46}
```

## Summary

In this notebook you learned how to:
- Create and use the Genova k-mer tokenizer
- Build a Genova transformer model
- Extract DNA sequence embeddings
- Predict variant effects by comparing ref/alt embeddings
- Use the InferenceEngine for production-style inference

Next: See `02_training.ipynb` for model training.